In [1]:
words = open('names.txt', 'r').read().splitlines()
b = {}
for word in words:
  word_pre_suf = ['<'] + list(word) + [']']
  for i in range(len(word_pre_suf) - 2):
    bigram = word_pre_suf[i] + word_pre_suf[i + 1]
    b[bigram] = b.get(bigram, 0) + 1

b = sorted(b.items(), key=lambda x: x[1], reverse=True)
print(b)

[('an', 5438), ('<a', 4410), ('ar', 3264), ('el', 3248), ('ri', 3033), ('na', 2977), ('<k', 2963), ('le', 2921), ('en', 2675), ('la', 2623), ('ma', 2590), ('<m', 2538), ('al', 2528), ('li', 2480), ('ia', 2445), ('<j', 2422), ('on', 2411), ('ra', 2356), ('ah', 2332), ('ha', 2244), ('ya', 2143), ('in', 2126), ('<s', 2055), ('ay', 2050), ('er', 1958), ('nn', 1906), ('yn', 1826), ('ka', 1731), ('ni', 1725), ('re', 1697), ('<d', 1690), ('ie', 1653), ('ai', 1650), ('<r', 1639), ('am', 1634), ('ly', 1588), ('<l', 1572), ('<c', 1542), ('<e', 1531), ('ja', 1473), ('ne', 1359), ('ll', 1345), ('il', 1345), ('is', 1316), ('<t', 1308), ('<b', 1306), ('da', 1303), ('sh', 1285), ('de', 1283), ('ee', 1271), ('mi', 1256), ('sa', 1201), ('<n', 1146), ('as', 1118), ('yl', 1104), ('ey', 1070), ('or', 1059), ('ad', 1042), ('ta', 1027), ('<z', 929), ('vi', 911), ('ke', 895), ('se', 884), ('<h', 874), ('ro', 869), ('es', 861), ('za', 860), ('ir', 849), ('br', 842), ('av', 834), ('me', 818), ('ei', 818), ('ca

In [2]:
import torch
N = torch.zeros((27, 27), dtype=torch.int32)

sorted_chars = sorted(list(set(''.join(words))))
enumerated = enumerate(sorted_chars)
char_to_idx = {char: idx + 1 for idx, char in enumerated}
char_to_idx['*'] = 0
idx_to_char = {idx: char for char, idx in char_to_idx.items()}


for word in words:
  word = '*' + word + '*'
  for ch1, ch2 in zip(word, word[1:]):
    ix1 = char_to_idx[ch1]
    ix2 = char_to_idx[ch2]
    N[ix1, ix2] += 1


In [4]:
P = N.float()
P = (P + 1) / P.sum(1, keepdims=True)

In [42]:
# Creating and training a neural net
import torch.nn.functional as F
import matplotlib.pyplot as plt
xs = []
ys = []
for word in words:
  work_updated = ['*'] + list(word) + ['*']
  for ch1, ch2 in zip(work_updated, work_updated[1:]):
    ix1 = char_to_idx[ch1]
    ix2 = char_to_idx[ch2]
    xs.append(ix1)
    ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)


x_encoded = F.one_hot(xs, num_classes=27).float()
y_encoded = F.one_hot(ys, num_classes=27).float()


W = torch.randn((27, 27), generator=torch.Generator().manual_seed(2147481317), requires_grad=True)

In [59]:
for i in range(500):
  ## Forward pass
  matmul = x_encoded @ W

  counts = matmul.exp()
  probs = counts /counts.sum(1, keepdim=True)

  arr = torch.arange(len(xs))
  vectorised_loss = -(probs[arr, ys]).log().mean() + 0.01 * (W**2).mean()
  print(f"Loss {vectorised_loss}")

  ## Cross entropy version
  # cross_entropy_loss = F.cross_entropy(x_encoded @ W, ys)
  # print(f"Loss {cross_entropy_loss}")

  # W.grad = None
  # cross_entropy_loss.backward()

  ## Backward pass
  W.grad = None
  vectorised_loss.backward()

  W.data += -5 * W.grad


RuntimeError: The function 'nll_loss_forward' is not differentiable with respect to argument 'weight'. This input cannot have requires_grad True.

In [58]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
  start_char = '*'
  ix = char_to_idx[start_char]
  out = []
  while True:
    #x_encoded = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    matmul = (W[ix]).reshape(1, -1)
    #print(matmul)
    #print(matmul.shape)
    counts = matmul.exp()
    probs = counts / counts.sum(1, keepdim=True)
    ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
    out.append(idx_to_char[ix])
    if ix == 0:
      break
  print(''.join(out))

cexze*
mogh*
mikilezityha*
konimittain*
llayn*


In [4]:
## Seeding data for a trigram neural net
sorted_chars = sorted(list(set(''.join(words))))
enumerated = enumerate(sorted_chars)
char_to_idx = {char: idx + 1 for idx, char in enumerated}
char_to_idx['*'] = 0
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

sorted_chars.insert(0, '*')
counter = 0
trigram_idx_map = {}
idx_trigram_map = {}
for i in range(len(sorted_chars)):
  for j in range(len(sorted_chars)):
    trigram = sorted_chars[i] + sorted_chars[j]
    trigram_idx_map[trigram] = counter
    idx_trigram_map[counter] = trigram
    counter += 1


print(sorted_chars)
print(trigram_idx_map)
print(idx_trigram_map)

['*', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
{'**': 0, '*a': 1, '*b': 2, '*c': 3, '*d': 4, '*e': 5, '*f': 6, '*g': 7, '*h': 8, '*i': 9, '*j': 10, '*k': 11, '*l': 12, '*m': 13, '*n': 14, '*o': 15, '*p': 16, '*q': 17, '*r': 18, '*s': 19, '*t': 20, '*u': 21, '*v': 22, '*w': 23, '*x': 24, '*y': 25, '*z': 26, 'a*': 27, 'aa': 28, 'ab': 29, 'ac': 30, 'ad': 31, 'ae': 32, 'af': 33, 'ag': 34, 'ah': 35, 'ai': 36, 'aj': 37, 'ak': 38, 'al': 39, 'am': 40, 'an': 41, 'ao': 42, 'ap': 43, 'aq': 44, 'ar': 45, 'as': 46, 'at': 47, 'au': 48, 'av': 49, 'aw': 50, 'ax': 51, 'ay': 52, 'az': 53, 'b*': 54, 'ba': 55, 'bb': 56, 'bc': 57, 'bd': 58, 'be': 59, 'bf': 60, 'bg': 61, 'bh': 62, 'bi': 63, 'bj': 64, 'bk': 65, 'bl': 66, 'bm': 67, 'bn': 68, 'bo': 69, 'bp': 70, 'bq': 71, 'br': 72, 'bs': 73, 'bt': 74, 'bu': 75, 'bv': 76, 'bw': 77, 'bx': 78, 'by': 79, 'bz': 80, 'c*': 81, 'ca': 82, 'cb': 83, 'cc': 84, 'cd': 85, 'ce': 86, 'cf

In [23]:
# Creating a training a trigram neural net
import torch.nn.functional as F
xs = []
ys = []
for word in words:
  work_updated = ['*', '*'] + list(word) + ['*', '*']
  for ch1, ch2, ch3 in zip(work_updated, work_updated[1:], work_updated[2:]):
    ix = trigram_idx_map[ch1 + ch2]
    iy = char_to_idx[ch3]
    xs.append(ix)
    ys.append(iy)

xs = torch.tensor(xs)
ys = torch.tensor(ys)


xs_encoded = F.one_hot(xs, num_classes=729).float()


W = torch.randn((729, 27), generator=torch.Generator().manual_seed(2147481317), requires_grad=True)
print(xs_encoded.shape)
print(W.shape)

torch.Size([260179, 729])
torch.Size([729, 27])


In [24]:
## Gradient descent on trigram neural net

for i in range(500):
  ## Forward pass


  # counts_arr = []
  # for x in xs:
  #   count_row = W[x]
  #   counts_arr.append(count_row)
  # counts_tensor = torch.stack(counts_arr)


  # matmul = xs_encoded @ W

  # counts = matmul.exp()
  # probs = counts /counts.sum(1, keepdim=True)

  # arr = torch.arange(len(xs))
  # vectorised_loss = -(probs[arr, ys]).log().mean() + 0.01 * (W**2).mean()
  # print(f"Loss {vectorised_loss}")

  # Cross entropy version
  cross_entropy_loss = F.cross_entropy(xs_encoded @ W, ys) + 0.01 * (W**2).mean()
  print(f"Loss {cross_entropy_loss}")

  W.grad = None
  cross_entropy_loss.backward()

  ## Backward pass
  # W.grad = None
  # vectorised_loss.backward()

  W.data += -10 * W.grad

Loss 3.745356798171997
Loss 3.709883213043213
Loss 3.6756792068481445
Loss 3.6426122188568115
Loss 3.610602378845215
Loss 3.579606771469116
Loss 3.5496082305908203
Loss 3.520610809326172
Loss 3.492626905441284
Loss 3.4656736850738525
Loss 3.439767837524414
Loss 3.4149231910705566
Loss 3.391148328781128
Loss 3.368446111679077
Loss 3.346809148788452
Loss 3.326221466064453
Loss 3.3066561222076416
Loss 3.288072347640991
Loss 3.2704195976257324
Loss 3.253638744354248
Loss 3.2376646995544434
Loss 3.2224326133728027
Loss 3.207879066467285
Loss 3.1939432621002197
Loss 3.1805710792541504
Loss 3.167713165283203
Loss 3.1553261280059814
Loss 3.1433706283569336
Loss 3.1318130493164062
Loss 3.1206231117248535
Loss 3.109773635864258
Loss 3.099241018295288
Loss 3.0890040397644043
Loss 3.0790438652038574
Loss 3.0693435668945312
Loss 3.0598878860473633
Loss 3.0506627559661865
Loss 3.0416557788848877
Loss 3.03285551071167
Loss 3.0242528915405273
Loss 3.0158374309539795
Loss 3.0076005458831787
Loss 2.9995

In [25]:
# Name generation using trigram neural net
g = torch.Generator().manual_seed(2147481317)
for i in range(50):
  prev_char = '*'
  ix = trigram_idx_map['*' + prev_char]
  out = []
  while True:
    #x_encoded = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    matmul = (W[ix]).reshape(1, -1)
    #print(matmul)
    #print(matmul.shape)
    counts = matmul.exp()
    probs = counts / counts.sum(1, keepdim=True)
    next_char_idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
    next_char = idx_to_char[next_char_idx]
    ix = trigram_idx_map[prev_char + next_char]
    prev_char = next_char
    out.append(next_char)
    if next_char == '*':
      break
  print(''.join(out))

ely*
marisie*
nvjlclqsgovhmhdnjrbckbpmztteenvfsdsyn*
kolmpurnnel*
bekpqrzymfgeqaxlqfsktxtyrqhtrunjvqnvvan*
josefon*
jadimkzmzrnooxbskpcxvomtplrhryeqtcgrcdckfwnrhvslp*
kaikaysrkamdorddkdor*
jalcxjruquby*
mepz*
noqdlnxvrtepfxbvewdwrjcirezdbixyilewzzwjzyostsamixkloltcbfhlmal*
sonyyueliahnuffpgqfomordlxqzzjbqkylwha*
hari*
ivcvy*
kbixmdovjobeoyhsugvjpluranahztbvy*
bry*
al*
anssrmrrosanain*
ziafainesmylasenna*
amni*
mi*
kwfzotvynah*
lyvqogrczuttgnyhcgjfywhnthjdnsiudtbrbottggy*
arkare*
ran*
jan*
milivericybqsfa*
ofvwtqoy*
stndri*
abeaodgtdaxevqkqsharnxpkxexitnyxppwvcbnkcrgah*
reicvmqjyqqqxvycwebihvne*
samijvklfvkvedeuwgdynn*
bey*
tahwsmvg*
xjfzqyovecjxwsmcz*
ancjani*
ady*
na*
epidtbucwvfotzixlfvxxfyiwrbasequitqlpile*
malimqgosjbxzgez*
kfmono*
yona*
dybcnldricvwuakalvfiio*
ariel*
na*
son*
pablnk*
la*
kacxqzhuton*
cpnror*
